# Phase 1: modules and infrastructure

This notebook checks the model implementation against protocol section 4.8,
measures the fp16 determinism path, runs the registered exploratory kernel
study, and measures runtime. The model tests run on any machine. The kernel
study and the smoke test need the GPU pod.

Finish condition: nine model tests pass, eight kernel runs succeed with
inner-validation scores only, and the runtime summary holds measured values.


In [1]:
%run definitions.ipynb
require_pythonhashseed()
PH1 = PROJECT_ROOT / "phase1" / "outputs"
PH1.mkdir(parents=True, exist_ok=True)
ENV = environment_record()
SPLITS_CTX = load_split_tables()
print(json.dumps({k: ENV[k] for k in ["timestamp_utc", "python", "definitions_sha256"]}, indent=2))


required packages already match the pins.
torch 2.8.0+cu128 present (training phases need >= 2.8.0).
torchvision 0.23.0+cu128 present (training phases need >= 0.23.0).


python 3.12.3 | numpy 2.2.6 | pandas 2.3.3 | pillow 12.0.0 | opencv 4.12.0 | torch available: True


{
  "timestamp_utc": "2026-08-23T21:04:09Z",
  "python": "3.12.3",
  "definitions_sha256": "cfcf3a434b8e82ce21ef86fdf1f548d338cab1ee53a6fd1d81d91bdf8d89a813"
}


In [2]:
# Registered model tests (protocol section 4.8).
test_table, param_counts = run_registered_model_tests()
print(test_table.to_string(index=False))
print()
print(pd.Series(param_counts).to_string())
write_json(PH1 / "model_tests.json",
           {"tests": {row["test"]: bool(row["pass"]) for _, row in test_table.iterrows()},
            "parameter_counts": param_counts, "environment": ENV})
print("\nall nine model tests pass")


                            test  pass
    1 adapter shapes equal input  True
 2 adapters are identity at init  True
           3 necks equal at init  True
     4 sinkhorn sums within 1e-5  True
          5 no backbone gradient  True
6 no batch-norm statistic change  True
   7 registered parameter counts  True
     8 adapter mismatch under 2%  True
  9 capacity mismatch under 0.1%  True

standard_adapter     76837
dka                  75856
residual_neck       134400
mhc_neck            232812
capacity_neck       232896

all nine model tests pass


In [3]:
# fp16 determinism smoke test (RUNBOOK: run before training on the target GPU).
if TORCH_AVAILABLE and torch.cuda.is_available():
    run_mixed_precision_determinism_smoke_test()
else:
    print("PENDING: the smoke test needs the GPU pod.")


smoke test passed, loss 0.719686 reproduced exactly


In [4]:
# Exploratory kernel study: large kernels 7, 15, 31, 51 at 50 and 1,044 images,
# fold 1, seed 42, DKA plus same-width residual neck, inner-validation only.
# The result cannot change the registered primary kernels.
KERNEL_STATUS = execute_pending_runs(PH1, kernel_study_specs(), SPLITS_CTX,
                                     "phase1.ipynb", export_test=False)


[1/8] p1_kernel7_n50


[2/8] p1_kernel7_n1044


[3/8] p1_kernel15_n50


[4/8] p1_kernel15_n1044


[5/8] p1_kernel31_n50


[6/8] p1_kernel31_n1044


[7/8] p1_kernel51_n50


[8/8] p1_kernel51_n1044


In [5]:
if KERNEL_STATUS == "complete":
    rows = []
    for spec in kernel_study_specs():
        meta = json.load(open(PH1 / "runs" / spec["run_id"] / "meta.json"))
        rows.append({"large_kernel": spec["dka_large"], "n": spec["n"],
                     "inner_macro_auc": meta["best_inner_macro_auc"],
                     "wall_seconds": meta["wall_seconds"],
                     "peak_gpu_gb": round(meta["peak_gpu_bytes"] / 1e9, 2)})
        test_file = PH1 / "runs" / spec["run_id"] / "predictions_test.npz"
        if test_file.exists():
            raise PhaseStop(f"{spec['run_id']} exported outer-test predictions.")
    kernel_table = pd.DataFrame(rows).sort_values(["n", "large_kernel"])
    kernel_table.to_csv(PH1 / "kernel_study.csv", index=False)
    print(kernel_table.to_string(index=False))
    print("\nno kernel run exported outer-test predictions")
else:
    print("PENDING: kernel table waits for the GPU runs.")


 large_kernel    n  inner_macro_auc  wall_seconds  peak_gpu_gb
            7   50         0.784448          36.0          0.6
           15   50         0.785150          35.6          0.6
           31   50         0.785184          35.8          0.6
           51   50         0.784955          40.8          0.6
            7 1044         0.913588         508.3          0.6
           15 1044         0.910951         509.5          0.6
           31 1044         0.913661         513.3          0.6
           51 1044         0.907544         463.1          0.6

no kernel run exported outer-test predictions


In [6]:
if KERNEL_STATUS == "complete":
    kt = pd.read_csv(PH1 / "kernel_study.csv")
    t50 = float(kt.loc[kt["n"] == 50, "wall_seconds"].mean())
    t1044 = float(kt.loc[kt["n"] == 1044, "wall_seconds"].mean())
    slope = (t1044 - t50) / (1044 - 50)
    fixed = t50 - slope * 50
    model_minutes = {"fixed_minutes": round(fixed / 60, 2),
                     "minutes_at_1044": round((fixed + slope * 1044) / 60, 2)}
    summary = {"measured_seconds_mean": {"n50": t50, "n1044": t1044},
               "linear_time_model": model_minutes,
               "provisional_model_minutes": {"fixed": 6, "at_1044": 38},
               "gpu": ENV.get("gpu", "unknown")}
    write_json(PH1 / "runtime_summary.json", summary)
    print(json.dumps(summary, indent=2))
else:
    print("PENDING: runtime summary waits for the GPU runs.")


{
  "measured_seconds_mean": {
    "n50": 37.05,
    "n1044": 498.54999999999995
  },
  "linear_time_model": {
    "fixed_minutes": 0.23,
    "minutes_at_1044": 8.31
  },
  "provisional_model_minutes": {
    "fixed": 6,
    "at_1044": 38
  },
  "gpu": "NVIDIA A100-SXM4-80GB"
}


In [7]:
tests_ok = json.load(open(PH1 / "model_tests.json"))
all_tests = all(tests_ok["tests"].values())
if KERNEL_STATUS == "complete" and all_tests and (PH1 / "runtime_summary.json").exists():
    print("PHASE 1 COMPLETE")
elif all_tests:
    print("model tests pass. PHASE 1 GPU STAGE PENDING: run this notebook on the pod.")
else:
    raise PhaseStop("model tests failed.")


PHASE 1 COMPLETE
